# SIFLOW · run_8 · DiffusionGemma train + eval + final paper artifacts (SIFLOW-G)

Trains the head-only SIFLOW-G student on the DiffusionGemma backbone and evaluates it, then regenerates ALL tables + figures from every results JSON collected so far. Copy `paper/tables_auto.tex` and `paper/figures/` back into the paper and recompile.

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** run_7 (gemma data + weights); ideally run_3/4/6 results on Drive too

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

In [ ]:
!python scripts/train.py --config siflow/config/gemma.yaml --set \
    data.tokens_path={BASE}/data/gemma_256.npy \
    out_dir={BASE}/ckpt/gemma run_id=siflow_gemma train.total_steps=12000

In [ ]:
!python scripts/evaluate.py --config siflow/config/gemma.yaml --system siflow \
    --ckpt-dir {BASE}/ckpt/gemma --ref-tokens {BASE}/data/gemma_val.npy \
    --n-samples 400 --k-list 1 4 --out results/gemma.json

In [ ]:
# Pull every result collected across sessions back from Drive, then regenerate everything
!mkdir -p results && cp -r {BASE}/results/* results/ 2>/dev/null || true
!python scripts/make_tables.py --results results
!python scripts/make_figures.py --results results --train-log {BASE}/ckpt/mdlm/train_log.jsonl
print(open("paper/tables_auto.tex").read())

In [ ]:
# --- Save outputs to Drive (so the next notebook can resume) ---
!cp -r results {BASE}/ 2>/dev/null || true
!cp -r paper/figures {BASE}/ 2>/dev/null || true
!cp -r paper/tables_auto.tex {BASE}/ 2>/dev/null || true
print('saved to', BASE)

**Done.** Drop `paper/tables_auto.tex` and `paper/figures/*.pdf` into the paper tree and recompile `paper/siflow_aaai.tex` — Tables 2–4 and the figures are now populated.